In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
import cv2
import os
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Conv2D, MaxPool2D, Dense, Flatten, Dropout

In [3]:
img_size=30
data=[]
labels=[]
train_path="/content/drive/MyDrive/traffic/Train"
test_csv = "/content/drive/MyDrive/traffic/Test.csv"
test_path = "/content/drive/MyDrive/traffic/Test"

In [4]:
NUM_CATEGORIES = len(os.listdir(train_path))
NUM_CATEGORIES

43

In [7]:
for i in range(NUM_CATEGORIES):
    path = os.path.join(train_path, str(i))
    images = os.listdir(path)
    for j in images:
        try:
            image = cv2.imread(os.path.join(path, j))
            image = cv2.resize(image, (img_size, img_size))
            data.append(image)
            labels.append(i)
        except Exception as e:
            print(e)

In [8]:
X = np.array(data) / 255.0
y = np.array(labels)

In [9]:
X.shape

(39225, 30, 30, 3)

In [13]:
y = to_categorical(y, NUM_CATEGORIES)

In [5]:
test_data = pd.read_csv(test_csv)

X_test = []
y_test = []
base_path = "/content/drive/MyDrive/traffic"

for i, row in test_data.iterrows():
    img_path = os.path.join(base_path, row['Path'])

    image = cv2.imread(img_path)
    if image is None:
        continue

    image = cv2.resize(image, (30, 30))
    X_test.append(image)
    y_test.append(row['ClassId'])

X_test = np.array(X_test) / 255.0
y_test = np.array(y_test)

In [11]:
y_test=to_categorical(y_test, NUM_CATEGORIES)

In [21]:
print(X_test.shape)
print(y_test.shape)

(11012, 30, 30, 3)
(11012, 43)


In [16]:

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(img_size,img_size,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),

    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(NUM_CATEGORIES, activation='softmax')
])

In [17]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [18]:
model.fit(
    X,
    y,
    epochs=20,
    validation_data=(X_test, y_test)
)

Epoch 1/20
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 66s 51ms/step - accuracy: 0.3431 - loss: 2.4387 - val_accuracy: 0.8585 - val_loss: 0.5619
Epoch 2/20
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 73s 59ms/step - accuracy: 0.8093 - loss: 0.6033 - val_accuracy: 0.9125 - val_loss: 0.3185
Epoch 3/20
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 67s 47ms/step - accuracy: 0.8758 - loss: 0.3816 - val_accuracy: 0.9377 - val_loss: 0.2516
Epoch 4/20
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 63s 51ms/step - accuracy: 0.9080 - loss: 0.2761 - val_accuracy: 0.9358 - val_loss: 0.2363
Epoch 5/20
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 63s 52ms/step - accuracy: 0.9271 - loss: 0.2263 - val_accuracy: 0.9445 - val_loss: 0.2227
Epoch 6/20
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 58s 47ms/step - accuracy: 0.9339 - loss: 0.1990 - val_accuracy: 0.9460 - val_loss: 0.2241
Epoch 7/20
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 56s 46ms/step - accuracy: 0.9464 - loss: 0.1584 - val_accuracy: 0.9510 - val_loss: 0.1993
Epoch 8/20
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 85s 48ms/step - accuracy: 0.9529 -

In [25]:
loss, acc = model.evaluate(X_test, y_test)
print(acc)

345/345 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9572 - loss: 0.2218
0.9572284817695618
